Da das mit der msa nicht geklappt hat, war der Plan jetzt über eine API nur die Sequnz der Genfamilien zu downloaden und damit dann erneut eine msa zu machen. Vermustlich handelt es sich bei den anderen Sequenzen um die ganze hc, weshalb die msa nicht erfolgreich war. 

folgender Code gibt mit Hilfe von anarci die variable domain der Sequez an.

In [90]:
from anarci import anarci

# Eingabesequenz
seq_name = "5wi9_entity3"
seq = "QVQLVESGGGVVQPGRSLRLSCAASGFTFSNYGIHWVRQAPGKGLEWVAVIWYDGSIKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDRAAAGLHYYYGMDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPK"



# ANARCI-Lauf mit IMGT-Schema
result = anarci([(seq_name, seq)], scheme='imgt')

# Ergebnisse entpacken
numbering = result[0][0][0]  # Liste von (position, aa)
domain_info = result[2][0]   # Metadaten: [header, domain1, domain2, ...]

# Gesuchte Domain
gesuchte_domain_id = "human_H"

# Domain finden
for domain in domain_info[1:]:  # Erste Zeile ist Header
    domain_id = domain[domain_info[0].index('id')]
    if domain_id == gesuchte_domain_id:
        start = int(domain[domain_info[0].index('query_start')])
        end = int(domain[domain_info[0].index('query_end')])
        print(f"✅ Gefunden: {domain_id} von Position {start} bis {end}")
        print(f"Variable Region:\n{seq[start:end+1]}")
        break
else:
    print("❌ Keine passende Domain gefunden.")

✅ Gefunden: human_H von Position 0 bis 124
Variable Region:
QVQLVESGGGVVQPGRSLRLSCAASGFTFSNYGIHWVRQAPGKGLEWVAVIWYDGSIKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDRAAAGLHYYYGMDVWGQGTTVTVSSA


hier habe ich nach dem Test das ganze auf alle Sequenzen der heavy chains aus der Datei heavy_chains_with_families.csv angewendet. 

In [102]:
import csv
from anarci import anarci

# === 1. FASTA-Datei lesen und in CSV umwandeln ===
fasta_path = "heavy_chains_with_families.fasta"
csv_path = "heavy_chains_with_families.csv"

entries = []
with open(fasta_path, "r") as fasta_file:
    name = None
    seq_lines = []
    for line in fasta_file:
        line = line.strip()
        if line.startswith(">"):
            if name and seq_lines:
                sequence = "".join(seq_lines)
                entries.append((name, sequence))
            name = line[1:].strip()  # Header ohne ">"
            seq_lines = []
        else:
            seq_lines.append(line)
    # Letzten Eintrag speichern
    if name and seq_lines:
        entries.append((name, "".join(seq_lines)))

# In CSV schreiben
with open(csv_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["seq_name", "sequence"])
    writer.writerows(entries)

print(f"✅ {len(entries)} Sequenzen in {csv_path} gespeichert.")

# === 2. CSV lesen und ANARCI anwenden ===
results = []

with open(csv_path, newline="") as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        seq_name = row["seq_name"].strip('"')  # Quotes entfernen, falls vorhanden
        sequence = row["sequence"]

        try:
            anarci_result = anarci([(seq_name, sequence)], scheme='imgt')
            numbering = anarci_result[0][0][0]
            domains = anarci_result[2][0]

            if len(domains) > 1:
                for domain in domains[1:]:
                    domain_id = domain[domains[0].index("id")]
                    start = int(domain[domains[0].index("query_start")])
                    end = int(domain[domains[0].index("query_end")])
                    variable_region = sequence[start:end+1]
                    results.append([seq_name, domain_id, start, end, variable_region])
            else:
                results.append([seq_name, "not_found", "", "", "domain_not_found"])

        except Exception as e:
            results.append([seq_name, "error", "", "", f"{e}"])

# === 3. Ergebnisse speichern ===
with open("anarci_results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["seq_name", "domain_id", "start", "end", "variable_region"])
    writer.writerows(results)

print(f"✅ Ergebnisse in anarci_results.csv gespeichert.")

✅ 2589 Sequenzen in heavy_chains_with_families.csv gespeichert.
✅ Ergebnisse in anarci_results.csv gespeichert.


In [110]:
import pandas as pd
from anarci import anarci

input_csv = "heavy_chains_with_families.csv"
output_csv = "cdr_output_from_numbering.csv"

df = pd.read_csv(input_csv)

output = []

for idx, row in df.iterrows():
    seq_id = row["seq_name"]
    sequence = row["sequence"]
    try:
        result = anarci([(seq_id, sequence)], scheme='imgt')
        numbering = result[0][0]  # Liste von (position, aa, annotation)
        # position ist ein Tupel wie (region_number, insertion) z.B. (31, '') für Position 31, oder (52, 'A') für 52A
        # annotation ist z.B. 'cdr1', 'fr1' etc.

        # CDR-Positionslisten sammeln
        cdr_positions = {"cdr1": [], "cdr2": [], "cdr3": []}

        for pos, aa, annotation in numbering:
            if annotation in cdr_positions:
                cdr_positions[annotation].append((pos, aa))

        # Aus den Positionen die Aminosäuren zusammensetzen (nach Positionsnummer sortieren)
        cdr_seqs = {}
        for cdr, aa_list in cdr_positions.items():
            # Sortiere nach Positionsnummer und Insertion (damit korrekt sortiert)
            aa_list_sorted = sorted(aa_list, key=lambda x: (x[0][0], x[0][1]))
            seq_cdr = "".join([aa for pos, aa in aa_list_sorted])
            cdr_seqs[cdr.upper()] = seq_cdr

        output.append({
            "seq_name": seq_id,
            **cdr_seqs
        })

    except Exception as e:
        print(f"⚠️ Fehler bei {seq_id}: {e}")
        output.append({
            "seq_name": seq_id,
            "CDR1": "error",
            "CDR2": "error",
            "CDR3": "error"
        })

output_df = pd.DataFrame(output)
output_df.to_csv(output_csv, index=False)

print(f"✅ Fertig! Ergebnisse gespeichert in {output_csv}")

⚠️ Fehler bei 6vmk_entity1|C,F,I,N,Q,T,W,X,a,d,g,j,m,p,s,v|IGHV1: 'NoneType' object is not iterable
⚠️ Fehler bei 6vmk_entity1|C,F,I,N,Q,T,W,X,a,d,g,j,m,p,s,v|IGHV1: 'NoneType' object is not iterable
⚠️ Fehler bei 6vmk_entity1|C,F,I,N,Q,T,W,X,a,d,g,j,m,p,s,v|IGHV1: 'NoneType' object is not iterable
⚠️ Fehler bei 6vmk_entity1|C,F,I,N,Q,T,W,X,a,d,g,j,m,p,s,v|IGHV1: 'NoneType' object is not iterable
⚠️ Fehler bei 6vmk_entity1|C,F,I,N,Q,T,W,X,a,d,g,j,m,p,s,v|IGHV1: 'NoneType' object is not iterable
⚠️ Fehler bei 6vmk_entity1|C,F,I,N,Q,T,W,X,a,d,g,j,m,p,s,v|IGHV1: 'NoneType' object is not iterable
⚠️ Fehler bei 6vmk_entity1|C,F,I,N,Q,T,W,X,a,d,g,j,m,p,s,v|IGHV1: 'NoneType' object is not iterable


KeyboardInterrupt: 

In [112]:
from anarci import anarci

seq_id = "8ivw_entity2|B,E,H,K|IGHV1"
sequence = "EVQLVQSGAEVKKPGATVKISCKVSGFNIKDYYIHWVQQAPGKGLEWMGRIDVEDDETKYAPKFQGRVTITADTSTDTAYMELSSLRSEDTAVYYCATPIYGSREAWFAYWGQGTLVTVSSASTKGPSVFPLAPCSRSTSESTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTKTYTCNVDHKPSNTKVDKRVGGSHHHHHH"

sequences = [(seq_id, sequence)]

results, _ = anarci(sequences, scheme="imgt", assign_germline=False)

if results and results[0][0] is not None:
    numbering = results[0][0]
    print(f"Nummerierung für {seq_id}:")
    for pos, aa, region in numbering:
        print(f"{pos}: {aa} ({region})")
else:
    print(f"Nummerierung für {seq_id} nicht möglich.")

ValueError: too many values to unpack (expected 2)